# Ring example

Reproduces the ring example results of Ashoff (2026), *Persistent Convolution: A Topological
Approach to Formal AI Alignment Testing* (DOI:
[10.18130/8k9j-9k42](https://doi.org/10.18130/8k9j-9k42)): mean representations with 95%
pointwise confidence bounds (top), energy statistics against the two-ring baseline with
significance stars (middle), and Wasserstein / JS distance heatmaps (bottom).

Three synthetic models, generated in-notebook with the original sampling scheme (2-D ring
samples, uniform angles, per-coordinate Gaussian noise; 150 points per model):

| model | rings (center, radius) | noise | points |
| --- | --- | --- | --- |
| 2 rings | (1.3, 2) r=0.5; (2, 0.75) r=0.4 | 0.0 | 75 per ring |
| 3 rings | + (0.75, 1) r=0.2 | 0.0 | 50 per ring |
| noisy rings | the 3-ring geometry | 0.1 | 50 per ring |

The two-ring model serves as the baseline for all comparisons. Reproduced figures are
statistically identical to the published ones but not pixel-identical (different bootstrap
draws than the original run).

In [ ]:
import numpy as np
import persiscope as ps


def sample_rings_2d(centers, radii, n_per_ring, noise_std=0.0, seed=12):
    """2-D ring samples: uniform angles, per-coordinate Gaussian noise."""
    rng = np.random.default_rng(seed)
    points = []
    for (cx, cy), r in zip(centers, radii):
        angles = rng.uniform(0.0, 2.0 * np.pi, size=n_per_ring)
        x = cx + r * np.cos(angles) + rng.normal(0.0, noise_std, size=n_per_ring)
        y = cy + r * np.sin(angles) + rng.normal(0.0, noise_std, size=n_per_ring)
        points.append(np.column_stack([x, y]))
    return np.vstack(points)


two_centers = [(1.3, 2.0), (2.0, 0.75)]
two_radii = [0.5, 0.4]
three_centers = two_centers + [(0.75, 1.0)]
three_radii = two_radii + [0.2]

two_rings = sample_rings_2d(two_centers, two_radii, n_per_ring=75)
three_rings = sample_rings_2d(three_centers, three_radii, n_per_ring=50)
noisy_rings = sample_rings_2d(three_centers, three_radii, n_per_ring=50, noise_std=0.1)

for name, pts in [("2 rings", two_rings), ("3 rings", three_rings), ("noisy rings", noisy_rings)]:
    print(f"{name:12s} {pts.shape}")

## Fit topological representations

Analysis parameters match the original run: $H_0$ homology, diagram rotation
$\theta = -3\pi/8$ with scaling $\alpha = \sqrt{2}/2$, tenting resolution 1000, landscape
order 0, silhouette power 0.5, and 30 bootstrap samples at 80% subsampling.

In [ ]:
tf = ps.TopologicalTransformer(
    homology_dim=0,
    theta=-3 * np.pi / 8,
    tenting_resolution=1000,
    landscape_order=0,
    silhouette_power=0.5,
    n_bootstrap=30,
    subsample=0.8,
    weight_method="euclidean",
    random_state=0,
)
reps = [
    tf.fit_transform(x, label=lab)
    for x, lab in [(two_rings, "2 rings"), (three_rings, "3 rings"), (noisy_rings, "noisy rings")]
]
reps

## Baseline report

Every model is compared to the two-ring baseline with 200 permutations per test.

In [ ]:
fig = ps.viz.plot_baseline_report(
    reps,
    baseline=0,                          # the two-ring model
    curve_metrics=("wasserstein", "js"),
    n_permutations=200,
    random_state=0,
)

`noisy rings` is by far the most different from the baseline: ring count and spacing noise both
change the $H_0$ merge structure. `3 rings` shows a smaller but significant separation, and the
baseline column reads `ns` by construction.